# Spans of Control Analysis

## Purpose
Assess manager-to-employee ratios and identify structural inefficiencies in the organization.

## Key Metrics
- **Span of Control**: Number of direct reports per manager (optimal: 5-9)
- **Organizational Layers**: Levels between CEO and frontline employees
- **Manager-to-IC Ratio**: Proportion of managers in workforce
- **Distance from CEO**: Average organizational distance (fewer layers = faster)

## Research Foundation
- Optimal spans research (Theobald & Nicholson-Crotty, 2005)
- Organizational design principles (Galbraith, 2014)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Generate data if needed
if not os.path.exists('../data/org_hierarchy.csv'):
    print("Generating sample data...")
    exec(open('../data/generate_sample_data.py').read())
else:
    print("Loading existing data...")

# Load datasets
org_hierarchy = pd.read_csv('../data/org_hierarchy.csv')
dept_metrics = pd.read_csv('../data/department_metrics.csv')

print(f"\nLoaded {len(org_hierarchy)} employees")
print(f"Loaded metrics for {len(dept_metrics)} departments")

## 1. Overall Organizational Structure

High-level assessment of organizational design.

In [ ]:
print(f"{'='*80}")
print(f"ORGANIZATIONAL STRUCTURE OVERVIEW")
print(f"{'='*80}")

# Calculate key metrics
total_employees = len(org_hierarchy)
total_managers = len(org_hierarchy[org_hierarchy['reports_count'] > 0])
total_ics = len(org_hierarchy[org_hierarchy['reports_count'] == 0])
manager_ratio = total_managers / total_employees * 100

max_layers = org_hierarchy['distance_from_ceo'].max()
avg_distance = org_hierarchy['distance_from_ceo'].mean()

print(f"\nWorkforce Composition:")
print(f"  Total employees:     {total_employees}")
print(f"  Managers:            {total_managers} ({manager_ratio:.1f}%)")
print(f"  Individual contributors: {total_ics} ({total_ics/total_employees*100:.1f}%)")

print(f"\nOrganizational Depth:")
print(f"  Maximum layers (CEO to IC): {max_layers}")
print(f"  Average distance from CEO:  {avg_distance:.1f}")

# Benchmark comparison
print(f"\n📊 BENCHMARK COMPARISON:")
if manager_ratio > 20:
    assessment = "⚠ HIGH - Too many managers (target: 12-18%)"
elif manager_ratio > 15:
    assessment = "→ MODERATE - Slightly manager-heavy"
else:
    assessment = "✓ LEAN - Healthy manager ratio"
print(f"  Manager ratio: {assessment}")

if max_layers > 7:
    layer_assessment = "⚠ TOO MANY LAYERS - Slows decision-making"
elif max_layers > 5:
    layer_assessment = "→ MODERATE - Consider flattening"
else:
    layer_assessment = "✓ FLAT - Fast decision-making structure"
print(f"  Organizational layers: {layer_assessment}")

print(f"\n{'='*80}")

# Visualize org structure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Headcount by level
level_counts = org_hierarchy['level'].value_counts()
level_order = ['C-Suite', 'VP', 'Director', 'Senior Manager', 'Manager', 'IC5', 'IC4', 'IC3', 'IC2', 'IC1']
level_counts = level_counts.reindex([l for l in level_order if l in level_counts.index])

colors_pyramid = ['#8b0000', '#d62728', '#ff7f0e', '#ffbb78', '#98df8a', 
                  '#2ca02c', '#1f77b4', '#aec7e8', '#c5b0d5', '#c49c94']

bars = ax1.barh(level_counts.index, level_counts.values, 
                color=colors_pyramid[:len(level_counts)], alpha=0.8, edgecolor='black')
ax1.set_xlabel('Number of Employees', fontsize=11, fontweight='bold')
ax1.set_title('Organizational Pyramid', fontsize=13, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

# Add value labels
for i, (level, count) in enumerate(level_counts.items()):
    ax1.text(count + 2, i, f'{count}', va='center', fontsize=10, fontweight='bold')

# Distribution by distance from CEO
distance_counts = org_hierarchy['distance_from_ceo'].value_counts().sort_index()

bars2 = ax2.bar(distance_counts.index, distance_counts.values, 
                alpha=0.7, edgecolor='black', color='steelblue')
ax2.set_xlabel('Distance from CEO (Layers)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Number of Employees', fontsize=11, fontweight='bold')
ax2.set_title('Employee Distribution by Organizational Layer', fontsize=13, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bar in bars2:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{int(height)}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 2. Span of Control Analysis

Assess whether managers have optimal number of direct reports.

In [ ]:
# Filter to managers only
managers = org_hierarchy[org_hierarchy['reports_count'] > 0].copy()

print(f"{'='*80}")
print(f"SPAN OF CONTROL ANALYSIS")
print(f"{'='*80}")

# Overall span statistics
avg_span = managers['reports_count'].mean()
median_span = managers['reports_count'].median()
min_span = managers['reports_count'].min()
max_span = managers['reports_count'].max()

print(f"\nSpan of Control Statistics:")
print(f"  Average:  {avg_span:.1f}")
print(f"  Median:   {median_span:.0f}")
print(f"  Range:    {min_span} - {max_span}")
print(f"  Target:   5-9 (optimal range)")

# Categorize spans
managers['span_category'] = pd.cut(
    managers['reports_count'],
    bins=[0, 4, 9, 100],
    labels=['Too Narrow (<5)', 'Optimal (5-9)', 'Too Wide (>9)']
)

span_dist = managers['span_category'].value_counts()
span_pcts = (span_dist / len(managers) * 100).round(1)

print(f"\nSpan Distribution:")
for category in ['Too Narrow (<5)', 'Optimal (5-9)', 'Too Wide (>9)']:
    if category in span_dist.index:
        count = span_dist[category]
        pct = span_pcts[category]
        marker = "⚠" if category != 'Optimal (5-9)' else "✓"
        print(f"  {marker} {category:20s}: {count:3d} managers ({pct:5.1f}%)")

# Issues count
narrow_spans = len(managers[managers['reports_count'] < 5])
wide_spans = len(managers[managers['reports_count'] > 9])

print(f"\n⚠ STRUCTURAL ISSUES:")
print(f"  Managers with too few reports (<5):  {narrow_spans} (inefficient)")
print(f"  Managers with too many reports (>9): {wide_spans} (overloaded)")

# Span by level
print(f"\n{'='*80}")
print(f"SPAN OF CONTROL BY MANAGEMENT LEVEL")
print(f"{'='*80}")

mgr_levels = ['C-Suite', 'VP', 'Director', 'Senior Manager', 'Manager']
span_by_level = managers[managers['level'].isin(mgr_levels)].groupby('level')['reports_count'].agg(['mean', 'median', 'min', 'max', 'count'])
span_by_level = span_by_level.reindex([l for l in mgr_levels if l in span_by_level.index])
span_by_level = span_by_level.round(1)

print(span_by_level)

print(f"\n{'='*80}")

# Visualize spans
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Histogram of spans
ax1.hist(managers['reports_count'], bins=range(1, max_span+2), 
            alpha=0.7, edgecolor='black', color='coral')
ax1.axvline(x=5, color='green', linestyle='--', linewidth=2, alpha=0.7, label='Optimal Range')
ax1.axvline(x=9, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax1.axvline(x=avg_span, color='red', linestyle='-', linewidth=2, label=f'Avg ({avg_span:.1f})')
ax1.set_xlabel('Number of Direct Reports', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Managers', fontsize=11, fontweight='bold')
ax1.set_title('Distribution of Spans of Control', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Box plot by level
level_data = [managers[managers['level'] == level]['reports_count'].values 
              for level in mgr_levels if level in managers['level'].values]
level_labels = [level for level in mgr_levels if level in managers['level'].values]

bp = ax2.boxplot(level_data, labels=level_labels, patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('lightblue')

# Add optimal range shading
ax2.axhspan(5, 9, alpha=0.2, color='green', label='Optimal Range (5-9)')

ax2.set_ylabel('Number of Direct Reports', fontsize=11, fontweight='bold')
ax2.set_title('Span of Control by Management Level', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Department-Level Analysis

Compare organizational efficiency across departments.

In [ ]:
print(f"{'='*80}")
print(f"DEPARTMENT ORGANIZATIONAL METRICS")
print(f"{'='*80}")

# Display department metrics
dept_display = dept_metrics[[
    'department', 'total_employees', 'total_managers', 'avg_span_of_control', 
    'manager_to_employee_ratio', 'avg_distance_from_ceo'
]].copy()

dept_display = dept_display.sort_values('avg_span_of_control')
print(dept_display.to_string(index=False))

# Identify departments needing restructuring
print(f"\n{'='*80}")
print(f"DEPARTMENTS NEEDING ATTENTION")
print(f"{'='*80}")

low_span_depts = dept_metrics[dept_metrics['avg_span_of_control'] < 5]
if len(low_span_depts) > 0:
    print(f"\n⚠ LOW SPANS (Manager-Heavy Departments):")
    for _, dept in low_span_depts.iterrows():
        print(f"  • {dept['department']:20s}: Avg span = {dept['avg_span_of_control']:.1f}")
        print(f"      {dept['total_managers']} managers for {dept['total_employees']} employees")
        print(f"      → Consider consolidating manager roles")

high_span_depts = dept_metrics[dept_metrics['avg_span_of_control'] > 9]
if len(high_span_depts) > 0:
    print(f"\n⚠ HIGH SPANS (Manager-Starved Departments):")
    for _, dept in high_span_depts.iterrows():
        print(f"  • {dept['department']:20s}: Avg span = {dept['avg_span_of_control']:.1f}")
        print(f"      {dept['total_managers']} managers for {dept['total_employees']} employees")
        print(f"      → Consider adding manager positions")

print(f"\n{'='*80}")

# Visualize department comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Span of control by department
dept_sorted = dept_metrics.sort_values('avg_span_of_control')
colors_dept = ['red' if x < 5 else 'orange' if x > 9 else 'green' 
               for x in dept_sorted['avg_span_of_control']]

bars1 = ax1.barh(dept_sorted['department'], dept_sorted['avg_span_of_control'], 
                 color=colors_dept, alpha=0.7, edgecolor='black')
ax1.axvline(x=5, color='green', linestyle='--', linewidth=2, alpha=0.5, label='Optimal Range')
ax1.axvline(x=9, color='green', linestyle='--', linewidth=2, alpha=0.5)
ax1.set_xlabel('Average Span of Control', fontsize=11, fontweight='bold')
ax1.set_title('Span of Control by Department', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Add value labels
for i, (idx, row) in enumerate(dept_sorted.iterrows()):
    ax1.text(row['avg_span_of_control'] + 0.2, i, f"{row['avg_span_of_control']:.1f}", 
            va='center', fontsize=9, fontweight='bold')

# Manager ratio by department
dept_sorted2 = dept_metrics.sort_values('manager_to_employee_ratio')
manager_pcts = dept_sorted2['manager_to_employee_ratio'] * 100

bars2 = ax2.barh(dept_sorted2['department'], manager_pcts, 
                 alpha=0.7, edgecolor='black', color='steelblue')
ax2.axvline(x=15, color='orange', linestyle='--', linewidth=2, alpha=0.7, label='Target (15%)')
ax2.set_xlabel('Manager-to-Employee Ratio (%)', fontsize=11, fontweight='bold')
ax2.set_title('Manager Density by Department', fontsize=13, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

# Add value labels
for i, (idx, pct) in enumerate(zip(dept_sorted2.index, manager_pcts)):
    ax2.text(pct + 0.5, i, f"{pct:.1f}%", 
            va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Restructuring Recommendations

Identify specific managers and teams that need adjustment.

In [ ]:
print(f"{'='*80}")
print(f"RESTRUCTURING RECOMMENDATIONS")
print(f"{'='*80}")

# Managers with problematic spans
too_narrow = managers[managers['reports_count'] < 4]
too_wide = managers[managers['reports_count'] > 10]

print(f"\n⚠ MANAGERS WITH TOO FEW REPORTS (<4):")
print(f"  Count: {len(too_narrow)}")
if len(too_narrow) > 0:
    print(f"\n  Top issues:")
    for _, mgr in too_narrow.nsmallest(5, 'reports_count').iterrows():
        print(f"    • {mgr['level']:15s} in {mgr['department']:15s}: {mgr['reports_count']} reports")
    
    print(f"\n  Recommendation: Consolidate or eliminate {len(too_narrow)} manager positions")
    print(f"  Estimated savings: ${len(too_narrow) * 150000:,.0f}/year (assume $150K avg manager salary)")

print(f"\n⚠ MANAGERS WITH TOO MANY REPORTS (>10):")
print(f"  Count: {len(too_wide)}")
if len(too_wide) > 0:
    print(f"\n  Top issues:")
    for _, mgr in too_wide.nlargest(5, 'reports_count').iterrows():
        print(f"    • {mgr['level']:15s} in {mgr['department']:15s}: {mgr['reports_count']} reports")
    
    print(f"\n  Recommendation: Add {len(too_wide)} manager positions or redistribute reports")
    print(f"  Estimated cost: ${len(too_wide) * 150000:,.0f}/year (new manager salaries)")

# Net impact
net_managers = len(too_wide) - len(too_narrow)
net_cost = net_managers * 150000

print(f"\n💰 NET IMPACT:")
if net_managers > 0:
    print(f"  Need to add {net_managers} net new managers")
    print(f"  Net cost increase: ${net_cost:,.0f}/year")
elif net_managers < 0:
    print(f"  Can eliminate {abs(net_managers)} net manager positions")
    print(f"  Net cost savings: ${abs(net_cost):,.0f}/year")
else:
    print(f"  Net neutral - redistribute existing managers")
    print(f"  Focus on rebalancing spans without adding/removing positions")

# Optimal structure proposal
print(f"\n{'='*80}")
print(f"OPTIMAL STRUCTURE PROPOSAL")
print(f"{'='*80}")

optimal_span = 7  # Target average
optimal_managers_needed = total_ics / optimal_span
current_frontline_managers = len(managers[managers['level'] == 'Manager'])

print(f"\nCurrent State:")
print(f"  ICs: {total_ics}")
print(f"  Frontline managers: {current_frontline_managers}")
print(f"  Average span: {total_ics/current_frontline_managers:.1f}")

print(f"\nOptimal State (target span = {optimal_span}):")
print(f"  ICs: {total_ics}")
print(f"  Frontline managers needed: {optimal_managers_needed:.0f}")
print(f"  Average span: {optimal_span}")

manager_adjustment = optimal_managers_needed - current_frontline_managers
print(f"\nAdjustment Needed: {manager_adjustment:+.0f} managers")

print(f"\n{'='*80}")

## Key Takeaways

1. **Optimal spans improve efficiency**: Managers with 5-9 reports are most effective
2. **Too many layers slow decisions**: Each layer adds communication overhead and delays
3. **Manager ratio matters**: >20% managers = inefficient; <12% = overloaded
4. **Department-specific issues**: Some teams are over-managed, others under-managed

**Recommended Next Steps**:
- Consolidate manager positions where spans are too narrow (<5)
- Add managers or redistribute reports where spans are too wide (>10)
- Target overall manager ratio of 12-15%
- Flatten organizational layers where possible (target ≤6 layers CEO to IC)